# Predicción de supervivencia en el Titanic

Esta versión corregida del notebook carga el dataset de forma robusta, limpia los datos, entrena un DecisionTree con separación train/test, muestra métricas y guarda el modelo. Se evita el uso de input() para que el notebook sea reproducible.


In [ ]:
# Librerías
import os
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1) Cargar el CSV (con fallback de rutas)
possible_paths = [
    'datos/titanic.csv',
    'titanic.csv',
    './titanic.csv'
]
datos = None
for p in possible_paths:
    if os.path.exists(p):
        datos = pd.read_csv(p)
        print(f'Leído desde: {p}')
        break

if datos is None:
    raise FileNotFoundError(
        "No se encontró 'titanic.csv'. Coloca el dataset en el mismo directorio que el notebook o en la carpeta 'datos/'."
    )

# Mostrar info básica
print('
Columnas disponibles:', list(datos.columns))
display(datos.head())

# 2) Selección y limpieza de columnas (segura)
cols = ['Pclass', 'Sex', 'Age']
# Si quieres añadir más columnas: 'SibSp','Parch','Fare','Embarked'
for c in cols:
    if c not in datos.columns:
        raise KeyError(f'Columna requerida no encontrada: {c}')

X = datos[cols].copy()
y = datos['Survived'].copy()

# Mapear Sex de forma segura
X['Sex'] = X['Sex'].map({'female': 0, 'male': 1})
if X['Sex'].isna().any():
    modo_sex = X['Sex'].mode().iloc[0] if not X['Sex'].mode().empty else 0
    X['Sex'] = X['Sex'].fillna(modo_sex)

# Rellenar Age con la mediana (más robusta que la media)
edad_mediana = X['Age'].median()
X['Age'] = X['Age'].fillna(edad_mediana)

# Asegurar tipos
X['Pclass'] = X['Pclass'].astype(int)

# 3) Separar train / test y entrenar modelo
mask = y.notna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Modelo con semilla y control de profundidad para evitar sobreajuste
ia = DecisionTreeClassifier(random_state=42, max_depth=5)
ia.fit(X_train, y_train)

# 4) Evaluación
y_pred = ia.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'
Accuracy en test: {acc:.4f}
')
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred, zero_division=0))

# 5) Predicción sobre un nuevo pasajero (ejemplo)
nuevo_pasajero = pd.DataFrame({
    'Pclass': [1],    # 1,2,3
    'Sex':   [0],     # 0 = female, 1 = male
    'Age':   [29]     # edad en años
})
pred = ia.predict(nuevo_pasajero)
print('
Nuevo pasajero:', nuevo_pasajero.to_dict(orient='records')[0])
print('Predicción (1 = sobreviviría, 0 = no sobreviviría):', int(pred[0]))

# 6) Guardar el modelo para uso posterior
model_path = 'decision_tree_titanic.joblib'
joblib.dump(ia, model_path)
print(f'Modelo guardado en {model_path}')
